# Notebook 1 - Pengumpulan Data Review Play Store

Notebook ini **hanya** berisi proses pengambilan data review dari Google Play Store secara mandiri dan penyimpanan raw data ke CSV.

Tahap preprocessing, labeling, dan pelatihan model dilakukan di `notebook_2_pelatihan_model.ipynb`.

In [1]:
%pip install -q google-play-scraper==1.2.4 pandas==2.2.2

Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  Preparing metadata (pyproject.toml) did not run successfully.
  exit code: 1
  
  [12 lines of output]
  + meson setup C:\Users\LENOVO\AppData\Local\Temp\pip-install-8tc5qyoi\pandas_3e4d651ec657484b8ea5fc7525893395 C:\Users\LENOVO\AppData\Local\Temp\pip-install-8tc5qyoi\pandas_3e4d651ec657484b8ea5fc7525893395\.mesonpy-jmnm4goy\build -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --vsenv --native-file=C:\Users\LENOVO\AppData\Local\Temp\pip-install-8tc5qyoi\pandas_3e4d651ec657484b8ea5fc7525893395\.mesonpy-jmnm4goy\build\meson-python-native-file.ini
  The Meson build system
  Version: 1.2.1
  Source dir: C:\Users\LENOVO\AppData\Local\Temp\pip-install-8tc5qyoi\pandas_3e4d651ec657484b8ea5fc7525893395
  Build dir: C:\Users\LENOVO\AppData\Local\Temp\pip-install-8tc5qyoi\pandas_3e4d651ec657484b8ea5fc7525893395\.mesonpy-jmnm4goy\build
  Build type: native build
  Project name: pandas
  Project version: 2.2.2
  
  ..\..\meson.build:2:0: ERROR: C

In [2]:
import pandas as pd
import time

try:
    from google_play_scraper import reviews, Sort
    SCRAPER_AVAILABLE = True
except ModuleNotFoundError as exc:
    print("Package google-play-scraper belum tersedia di environment ini:", exc)
    print("Jalankan: pip install -r requirements.txt, atau gunakan raw CSV hasil scraping yang disertakan.")
    SCRAPER_AVAILABLE = False
    reviews = None
    Sort = None

APP_ID = "com.indomaret.klikindomaret"
APP_NAME = "Klik Indomaret"
COUNTRY = "id"
LANGUAGE = "id"
COUNT_PER_SCORE = 1000
MIN_RAW_SAMPLE = 3000
RAW_OUTPUT = "raw_review_klikindomaret_playstore.csv"

In [3]:
all_reviews = []

if SCRAPER_AVAILABLE:
    try:
        for score in [1, 2, 3, 4, 5]:
            print(f"Mengambil review rating {score} ...")
            result, _ = reviews(
                APP_ID,
                lang=LANGUAGE,
                country=COUNTRY,
                sort=Sort.NEWEST,
                count=COUNT_PER_SCORE,
                filter_score_with=score
            )
            print(f"Rating {score}: {len(result)} review")
            all_reviews.extend(result)
            time.sleep(1)

        raw_df = pd.DataFrame(all_reviews)
        raw_df = raw_df.drop_duplicates(subset=["reviewId"]).reset_index(drop=True)
        raw_df.to_csv(RAW_OUTPUT, index=False)

    except Exception as exc:
        print("Scraping tidak dapat dijalankan pada environment ini:", exc)
        print("Memuat file raw CSV hasil scraping yang sudah disertakan.")
        raw_df = pd.read_csv(RAW_OUTPUT)
else:
    print("Memuat file raw CSV hasil scraping karena package scraper belum tersedia.")
    raw_df = pd.read_csv(RAW_OUTPUT)

print("Jumlah raw data hasil scraping:", len(raw_df))
print("Bentuk raw_df:", raw_df.shape)
raw_df.head()

Mengambil review rating 1 ...
Rating 1: 1000 review
Mengambil review rating 2 ...
Rating 2: 1000 review
Mengambil review rating 3 ...
Rating 3: 1000 review
Mengambil review rating 4 ...
Rating 4: 1000 review
Mengambil review rating 5 ...
Rating 5: 1000 review
Jumlah raw data hasil scraping: 5000
Bentuk raw_df: (5000, 11)


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,f10ee8dd-476f-4865-8ce1-6aa83b277e7b,Nurul Lantika,https://play-lh.googleusercontent.com/a-/ALV-U...,jngn di download nyesel,1,0,2401101,2026-05-13 18:16:06,None,NaT,2401101
1,0e9008f6-872b-4c64-b3dd-d4d2ab21a7ad,Andra Winata,https://play-lh.googleusercontent.com/a-/ALV-U...,kecewa order di Indomart skrg stok banyak koso...,1,0,2604200,2026-05-13 18:04:31,None,NaT,2604200
2,c139cef1-5566-4086-a072-51c2b8e3af61,Indri Aprilliani,https://play-lh.googleusercontent.com/a-/ALV-U...,aplikasinya -100/10 beda banget sama alfagift,1,0,None,2026-05-13 17:20:07,None,NaT,None
3,938c2a68-fb03-40a8-8e79-8bf236230203,Fina Zee,https://play-lh.googleusercontent.com/a-/ALV-U...,makan tuh sendiri.promo.gula.minyak.beras.majo...,1,0,2604200,2026-05-13 16:52:53,None,NaT,2604200
4,aae08446-1e26-4bbb-8f7e-2cc4a0b453d9,ismi rizqi kamila,https://play-lh.googleusercontent.com/a-/ALV-U...,tolong di respon dengan baik terkait proses pe...,1,0,2604200,2026-05-13 16:19:21,None,NaT,2604200


## Cek jumlah data raw

Nilai `RAW_COUNT_SEBELUM_PREPROCESSING` harus minimal 3000. Data pada tahap ini masih raw dan belum melalui preprocessing/labeling.

In [8]:
RAW_COUNT_SEBELUM_PREPROCESSING = len(raw_df)
print("RAW_COUNT_SEBELUM_PREPROCESSING =", RAW_COUNT_SEBELUM_PREPROCESSING)
print("Memenuhi minimal 3.000 data?", RAW_COUNT_SEBELUM_PREPROCESSING >= MIN_RAW_SAMPLE)
print("Distribusi rating raw:")
print(raw_df["score"].value_counts().sort_index())

RAW_COUNT_SEBELUM_PREPROCESSING = 5000
Memenuhi minimal 3.000 data? True
Distribusi rating raw:
score
1    1000
2    1000
3    1000
4    1000
5    1000
Name: count, dtype: int64
